# OWOD Replay Protocol V3 — one-click Colab experiment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gubiczam/owod-active/blob/main/notebooks/owod_active.ipynb)

Open in a fresh **GPU** runtime and choose **Runtime → Run all**. Google Drive
authorization is the only expected interaction. The notebook is pinned to reviewed OWL
and PROB commits, validates the completed no-replay baseline before training, resumes
safe partial runs, and publishes a strict baseline/uniform/tail comparison to Drive.

The scientific protocol is owned by `owl.runner`; these cells only prepare, verify,
orchestrate, audit, and report it. A failed assertion stops the expensive path.


In [ ]:
# 0 — Parameters and immutable experiment identity
# ============================== PARAMETERS ==============================
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

RUN_GPU = True
FAST_CHAIN = True
SELECTION_ARM = "random"
LABELLING_POLICY = "known_plus_selected"
N_TASKS = 6
BUDGET_PER_TASK = 600
ROUNDS_PER_TASK = 6
CANDIDATE_IMAGES = 2000
PROPOSALS_PER_IMAGE = 50
REPLAY_REALLOCATE = False
EPOCHS = 5
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
N_CLUSTERS = 1600
SEED = 0
REPLAY_ARMS = ("uniform", "tail_favouring")
EVAL_MAX_PER_CLASS = 150
EVAL_REMAINDER_RATIO = 1
TIME_BUDGET_MINUTES = 420
SESSION_CEILING_MINUTES = 840

OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"
OWL_COMMIT = "ae2d2ab1bdeb7a9c30992448d0a839c3458451e9"
PROB_REPOSITORY = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "4c66be1a52cad9360e09c729e9134aba8fe0b531"

DRIVE_ROOT = "/content/drive/MyDrive/OWL"
CHECKPOINT_RELATIVE = "checkpoints/SOWODB/t1.pth"
BASELINE_NAME = "random__none"
PLANNED_RUNS = ("random__uniform", "random__tail_favouring")
COMPARISON_NAME = "replay_v3_fast_seed0"
SESSION_STARTED = time.monotonic()
RUN_STATUS = {}
VALIDATION_OUTPUTS = {}

assert RUN_GPU and FAST_CHAIN
assert (N_TASKS, BUDGET_PER_TASK, ROUNDS_PER_TASK) == (6, 600, 6)
assert (CANDIDATE_IMAGES, PROPOSALS_PER_IMAGE) == (2000, 50)
assert (EPOCHS, LEARNING_RATE, BATCH_SIZE, N_CLUSTERS, SEED) == (5, 2e-4, 2, 1600, 0)
assert REPLAY_ARMS == ("uniform", "tail_favouring") and not REPLAY_REALLOCATE
EXPERIMENT_AUDIT = {
    "selection": SELECTION_ARM, "labelling": LABELLING_POLICY,
    "tasks": N_TASKS, "annotation_budget": BUDGET_PER_TASK,
    "rounds": ROUNDS_PER_TASK, "candidate_images": CANDIDATE_IMAGES,
    "proposals_per_image": PROPOSALS_PER_IMAGE,
    "replay_arms": REPLAY_ARMS, "replay_reallocate": REPLAY_REALLOCATE,
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE, "clusters": N_CLUSTERS, "seed": SEED,
    "per_run_minutes": TIME_BUDGET_MINUTES,
    "session_ceiling_minutes": SESSION_CEILING_MINUTES,
    "workspaces": (BASELINE_NAME, *PLANNED_RUNS),
}
print("Pinned experiment:", OWL_COMMIT[:12], PROB_COMMIT[:12])
print(json.dumps(EXPERIMENT_AUDIT, indent=2))


In [ ]:
# 1 — Mount Drive and prove the persistent root is writable
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
_drive_probe = DRIVE / ".owod_write_probe"
_drive_probe.write_text("ok", encoding="utf-8")
assert _drive_probe.read_text(encoding="utf-8") == "ok"
_drive_probe.unlink()
print("Drive writable:", DRIVE)


In [ ]:
# 2 — Pin OWL exactly, install its declared dependencies, and import fresh code
def _checked(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)


def _capture(command, **kwargs):
    return _checked(command, capture_output=True, **kwargs).stdout.strip()


def _normalise_git_url(value):
    value = value.strip().removesuffix(".git").rstrip("/")
    if value.startswith("git@github.com:"):
        value = "https://github.com/" + value.split(":", 1)[1]
    return value


def ensure_pinned_checkout(path, repository, commit):
    path = Path(path)
    expected = _normalise_git_url(repository)
    if path.exists():
        assert (path / ".git").is_dir(), f"Refusing non-git path: {path}"
        origin = _normalise_git_url(_capture(["git", "remote", "get-url", "origin"], cwd=path))
        assert origin == expected, f"Refusing unexpected origin at {path}: {origin}"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        _checked(["git", "clone", "--filter=blob:none", "--no-checkout", repository, str(path)])
    _checked(["git", "fetch", "--depth", "1", "origin", commit], cwd=path)
    _checked(["git", "reset", "--hard", commit], cwd=path)
    _checked(["git", "clean", "-fdx"], cwd=path)
    actual = _capture(["git", "rev-parse", "HEAD"], cwd=path)
    assert actual == commit, f"{path}: expected {commit}, got {actual}"
    return path


ROOT = ensure_pinned_checkout(Path("/content/owod-active"), OWL_REPOSITORY, OWL_COMMIT)
_checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
          "-e", f"{ROOT}[plots]"])

for _name in [n for n in sys.modules if n == "owl" or n.startswith("owl.")]:
    del sys.modules[_name]
sys.path.insert(0, str(ROOT))

from owl import bridge, comparison, evaluation_subset, exemplars, metrics, protocol, replay, runner

_required = {
    "run_chain.prepare_images": "prepare_images" in __import__("inspect").signature(runner.run_chain).parameters,
    "CycleConfig.replay_protocol_version": "replay_protocol_version" in runner.CycleConfig.__dataclass_fields__,
    "comparison.compatibility": hasattr(comparison, "compatibility"),
    "metrics.validate_per_class_ap50": hasattr(metrics, "validate_per_class_ap50"),
}
assert all(_required.values()), {k: v for k, v in _required.items() if not v}
OWL_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=ROOT)
print("OWL ready:", OWL_SHA, "from", ROOT)


In [ ]:
# 3 — Pin and validate the reviewed PROB bridge; build the optional CUDA kernel
PROB = ensure_pinned_checkout(Path("/content/PROB"), PROB_REPOSITORY, PROB_COMMIT)

# PROB's 2022 requirements file pins packages that have no Python 3.13 wheels
# (notably scikit-image 0.19.2 and pandas 1.5.1). The bridge does not import
# scikit-image, notebook, or ipdb. Install only its runtime imports, without
# replacing Colab's matched torch/torchvision/numpy stack. pycocotools stays
# at PROB's exact 2.0.5 pin: Cython generates the C source omitted by its sdist.
assert sys.version_info[:2] == (3, 13), sys.version
def distribution_version(distribution):
    probe = subprocess.run(
        [sys.executable, "-c",
         f"from importlib.metadata import version; print(version({distribution!r}))"],
        capture_output=True, text=True, check=False)
    return probe.stdout.strip() if probe.returncode == 0 else None


def module_available(module):
    return subprocess.run(
        [sys.executable, "-c", f"import {module}"],
        capture_output=True, text=True, check=False).returncode == 0


PROB_COMPAT_INSTALLED = []
if distribution_version("einops") != "0.5.0" or not module_available("einops"):
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "einops==0.5.0"])
    PROB_COMPAT_INSTALLED.append("einops==0.5.0")

if (distribution_version("pycocotools") != "2.0.5"
        or not module_available("pycocotools")):
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "Cython==3.1.3"])
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--no-build-isolation",
              "--no-deps", "--force-reinstall", "pycocotools==2.0.5"])
    PROB_COMPAT_INSTALLED.append("pycocotools==2.0.5")

# These imports are required transitively by main_open_world/engine. Keep a
# compatible Colab package when one is already importable; install a pinned
# Python-3.13 wheel only when it is absent or broken. WandB is disabled by the
# reviewed bridge and therefore cannot affect training or evaluation.
compatibility_wheels = {
    "wandb": "wandb==0.18.7",
    "pandas": "pandas==2.3.2",
    "seaborn": "seaborn==0.13.2",
    "tqdm": "tqdm==4.67.1",
}
missing_wheels = [spec for module, spec in compatibility_wheels.items()
                  if not module_available(module)]
if missing_wheels:
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              *missing_wheels])
    PROB_COMPAT_INSTALLED.extend(missing_wheels)

assert distribution_version("einops") == "0.5.0" and module_available("einops")
assert (distribution_version("pycocotools") == "2.0.5"
        and module_available("pycocotools"))


def pip_check():
    return subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        capture_output=True, text=True, check=False)


# Current Colab carries IPython metadata that requires Jedi while omitting
# Jedi itself. Repair only that observed metadata conflict, using a universal
# wheel with explicit Python 3.13 support, then require the complete package
# environment to pass the same check used by the final preflight.
bootstrap_package_probe = pip_check()
_package_conflicts = bootstrap_package_probe.stdout + bootstrap_package_probe.stderr
_missing_ipython_jedi = (
    "requires jedi, which is not installed" in _package_conflicts.lower()
    and "ipython " in _package_conflicts.lower()
)
if _missing_ipython_jedi:
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "jedi==0.19.2"])
    PROB_COMPAT_INSTALLED.append("jedi==0.19.2")
    bootstrap_package_probe = pip_check()
if bootstrap_package_probe.returncode != 0:
    print(bootstrap_package_probe.stdout + bootstrap_package_probe.stderr)
    raise RuntimeError("Python package consistency check failed after bootstrap repair")
print("Bootstrap package consistency: PASS")
runtime_probe = subprocess.run(
    [sys.executable, "-c",
     "import numpy, torch, torchvision, scipy, sklearn, PIL, matplotlib, pandas, seaborn, tqdm, wandb; "
     "from einops import rearrange; from pycocotools.coco import COCO; "
     "import main_open_world; from datasets.coco import make_coco_transforms; "
     "from datasets.torchvision_datasets.open_world import OWDetection; "
     "from engine import evaluate; from models import build_model"],
    cwd=PROB, capture_output=True, text=True, check=False)
if runtime_probe.returncode != 0:
    print(runtime_probe.stdout)
    print(runtime_probe.stderr)
    raise RuntimeError("Pinned PROB failed its Python runtime import probe")
print("PROB runtime imports: PASS; installed:", PROB_COMPAT_INSTALLED or "nothing")

# pycocotools 2.0.5 predates NumPy 2.0. Exercise PROB's own evaluator
# wrapper with the two removed aliases it needs, instead of accepting an
# import-only success. The aliases are confined to this fresh subprocess.
coco_smoke_code = r'''
import numpy as np
import torch
if "float" not in np.__dict__:
    np.float = float
if "NPY_OWNDATA" not in np.__dict__:
    np.NPY_OWNDATA = 4
from pycocotools.coco import COCO
from datasets.coco_eval import CocoEvaluator
coco = COCO()
coco.dataset = {
    "info": {}, "licenses": [],
    "images": [{"id": 1, "width": 32, "height": 32}],
    "categories": [{"id": 1, "name": "object", "supercategory": "object"}],
    "annotations": [{"id": 1, "image_id": 1, "category_id": 1,
                     "bbox": [4.0, 5.0, 10.0, 11.0], "area": 110.0, "iscrowd": 0}],
}
coco.createIndex()
evaluator = CocoEvaluator(coco, ("bbox",))
evaluator.update({1: {"boxes": torch.tensor([[4.0, 5.0, 14.0, 16.0]]),
                      "scores": torch.tensor([0.99]), "labels": torch.tensor([1])}})
evaluator.synchronize_between_processes()
evaluator.accumulate()
evaluator.summarize()
assert float(evaluator.coco_eval["bbox"].stats[0]) > 0.99
print("PROB pycocotools COCOeval smoke: PASS")
'''
coco_smoke = subprocess.run([sys.executable, "-c", coco_smoke_code], cwd=PROB,
                            capture_output=True, text=True, check=False)
if coco_smoke.returncode != 0:
    print(coco_smoke.stdout)
    print(coco_smoke.stderr)
    raise RuntimeError("Pinned pycocotools failed PROB's functional COCOeval smoke test")
print(coco_smoke.stdout.splitlines()[-1])


def run_json_probe(code, marker, *, cwd):
    probe = subprocess.run(
        [sys.executable, "-c", code], cwd=cwd,
        capture_output=True, text=True, check=False,
    )
    rows = [line.removeprefix(marker) for line in probe.stdout.splitlines()
            if line.startswith(marker)]
    if not rows:
        return {
            "probe_ok": False,
            "returncode": probe.returncode,
            "error": (probe.stderr or probe.stdout).strip() or "probe produced no result",
        }
    payload = json.loads(rows[-1])
    payload["returncode"] = probe.returncode
    return payload


# Deliberately diagnostic only: unlike PROB, this fresh interpreter does not
# import torch before loading the extension. On Colab that can fail to resolve
# PyTorch shared libraries even when PROB's real import and dispatch work.
raw_msda_probe_code = r"""
import importlib
import json
try:
    importlib.invalidate_caches()
    extension = importlib.import_module("MultiScaleDeformableAttention")
    payload = {"ok": True, "path": getattr(extension, "__file__", None), "error": None}
except BaseException as error:
    payload = {"ok": False, "path": None,
               "error": f"{type(error).__name__}: {error}"}
print("OWOD_RAW_MSDA_PROBE=" + json.dumps(payload, sort_keys=True))
"""
RAW_MSDA = run_json_probe(
    raw_msda_probe_code, "OWOD_RAW_MSDA_PROBE=", cwd=PROB.parent)

# Authoritative pre-build/post-build probe: import through the exact wrapper and
# downstream module used by PROB training. The pinned wrapper imports torch
# before the extension and the downstream module copies this boolean for dispatch.
prob_msda_probe_code = r"""
import importlib
import importlib.metadata
import json
import platform
import sys
from pathlib import Path
import einops
import matplotlib
import numpy
import pandas
import PIL
import pycocotools
import scipy
import sklearn
import torch
import torchvision

after_torch = {"ok": False, "path": None, "error": None}
try:
    importlib.invalidate_caches()
    extension = importlib.import_module("MultiScaleDeformableAttention")
    after_torch = {"ok": True, "path": getattr(extension, "__file__", None), "error": None}
except BaseException as error:
    after_torch["error"] = f"{type(error).__name__}: {error}"

payload = {
    "probe_ok": False,
    "python": platform.python_version(),
    "executable": sys.executable,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torch_cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "pillow": PIL.__version__,
    "matplotlib": matplotlib.__version__,
    "pandas": pandas.__version__,
    "einops": importlib.metadata.version("einops"),
    "pycocotools": importlib.metadata.version("pycocotools"),
    "extension_after_torch": after_torch,
}
try:
    from models.ops.functions import ms_deform_attn_func as msda_func
    from models.ops.modules import ms_deform_attn as msda_module
    expected_root = Path.cwd().resolve()
    wrapper_path = Path(msda_func.__file__).resolve()
    downstream_path = Path(msda_module.__file__).resolve()
    assert wrapper_path.is_relative_to(expected_root), wrapper_path
    assert downstream_path.is_relative_to(expected_root), downstream_path
    available = bool(msda_func.MSDA_AVAILABLE)
    assert bool(msda_module.MSDA_AVAILABLE) == available
    payload.update({
        "probe_ok": True,
        "available": available,
        "backend": "compiled" if available else "PyTorch fallback",
        "wrapper_path": str(wrapper_path),
        "downstream_path": str(downstream_path),
        "extension_path": (getattr(msda_func.MSDA, "__file__", None)
                           if available else None),
        "error": None,
    })
except BaseException as error:
    payload["error"] = f"{type(error).__name__}: {error}"
print("OWOD_PROB_MSDA_PROBE=" + json.dumps(payload, sort_keys=True))
"""


def probe_prob_msda():
    return run_json_probe(
        prob_msda_probe_code, "OWOD_PROB_MSDA_PROBE=", cwd=PROB)


PREBUILD_PROB_MSDA = probe_prob_msda()
_msda_fingerprint = json.dumps({
    "prob": PROB_COMMIT,
    "python": PREBUILD_PROB_MSDA.get("python", sys.version),
    "torch": PREBUILD_PROB_MSDA.get("torch"),
    "torch_cuda": PREBUILD_PROB_MSDA.get("torch_cuda"),
    "gpu": PREBUILD_PROB_MSDA.get("gpu"),
}, sort_keys=True)
MSDA_BUILD_MARKER = (
    PROB.parent / ".owod-active-cache" /
    f"msda-build-{hashlib.sha256(_msda_fingerprint.encode()).hexdigest()[:16]}.json"
)
MSDA_BUILD_ATTEMPTED = False
MSDA_BUILD_RETURN_CODE = None
if PREBUILD_PROB_MSDA.get("available") is not True:
    if MSDA_BUILD_MARKER.is_file():
        print("Skipping a previously built-but-unused MSDA extension for this exact runtime:",
              MSDA_BUILD_MARKER)
    else:
        if not module_available("ninja"):
            _checked([sys.executable, "-m", "pip", "install",
                      "--disable-pip-version-check", "-q", "ninja"])
        MSDA_BUILD_ATTEMPTED = True
        build = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--no-build-isolation", "--no-deps", "--force-reinstall", "."],
            cwd=PROB / "models" / "ops", text=True,
            capture_output=True, check=False,
        )
        MSDA_BUILD_RETURN_CODE = build.returncode
        if build.returncode != 0:
            print("WARNING: optional CUDA extension build failed; the full CUDA smoke "
                  "must prove PROB's PyTorch fallback.")
            print("\n".join((build.stdout + "\n" + build.stderr).splitlines()[-20:]))

POSTBUILD_PROB_MSDA = probe_prob_msda()
print("Raw extension import:", "PASS" if RAW_MSDA.get("ok") else "FAIL")
print("Raw extension path:", RAW_MSDA.get("path") or "unavailable")
if RAW_MSDA.get("error"):
    print("Raw extension error:", RAW_MSDA["error"])
print("PROB MSDA_AVAILABLE:", POSTBUILD_PROB_MSDA.get("available"))
print("PROB wrapper path:", POSTBUILD_PROB_MSDA.get("wrapper_path", "unavailable"))
print("PROB extension path:", POSTBUILD_PROB_MSDA.get("extension_path", "unavailable"))
if RAW_MSDA.get("ok") != POSTBUILD_PROB_MSDA.get("available"):
    print("MSDA diagnostic disagreement explained: the raw probe loads the extension "
          "before torch; pinned PROB imports torch first, then binds the extension "
          "inside models.ops.functions.ms_deform_attn_func. Only PROB's downstream "
          "dispatch is authoritative.")

# Run the real PROB builder and training loss on CUDA. This observes the exact
# MSDeformAttn branch taken by the model and requires that branch to participate
# in forward and backward before any experiment evaluation or training can run.
prob_smoke_code = r"""
import json
import sys
import tempfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
if "bool" not in np.__dict__:
    np.bool = np.bool_
import main_open_world
from datasets.coco import make_coco_transforms
from datasets.open_world_eval import voc_eval
from datasets.torchvision_datasets.open_world import OWDetection
from models import build_model
from models.ops.functions import ms_deform_attn_func as msda_func
from models.ops.modules import ms_deform_attn as msda_module
assert torch.cuda.is_available(), "CUDA is unavailable to the real PROB smoke test"
PROB_MSDA_AVAILABLE = bool(msda_func.MSDA_AVAILABLE)
assert bool(msda_module.MSDA_AVAILABLE) == PROB_MSDA_AVAILABLE
assert Path(msda_func.__file__).resolve().is_relative_to(Path.cwd().resolve())
assert Path(msda_module.__file__).resolve().is_relative_to(Path.cwd().resolve())
dispatch_counts = {"compiled": 0, "fallback": 0}
if PROB_MSDA_AVAILABLE:
    original_apply = msda_module.MSDeformAttnFunction.apply
    class ObservedCompiledDispatch:
        @staticmethod
        def apply(*arguments):
            dispatch_counts["compiled"] += 1
            return original_apply(*arguments)
    msda_module.MSDeformAttnFunction = ObservedCompiledDispatch
else:
    original_fallback = msda_module.ms_deform_attn_core_pytorch
    def observed_fallback(*arguments, **keywords):
        dispatch_counts["fallback"] += 1
        return original_fallback(*arguments, **keywords)
    msda_module.ms_deform_attn_core_pytorch = observed_fallback
args = main_open_world.get_args_parser().parse_args([])
args.device = "cuda"
args.dataset = "OWDETR"
args.PREV_INTRODUCED_CLS = 0
args.CUR_INTRODUCED_CLS = 20
args.num_classes = 81
args.model_type = "prob"
args.wandb_project = ""
args.wandb_name = ""
args.batch_size = 1
args.num_workers = 0
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    (root / "Annotations").mkdir()
    (root / "JPEGImages").mkdir()
    (root / "ImageSets" / "OWDETR").mkdir(parents=True)
    image_id = "000000000001"
    (root / "ImageSets" / "OWDETR" / "smoke_val.txt").write_text(image_id + "\n")
    Image.new("RGB", (32, 32), (0, 0, 0)).save(root / "JPEGImages" / f"{image_id}.jpg")
    annotation = ("<annotation><filename>000000000001.jpg</filename>"
                  "<size><width>32</width><height>32</height><depth>3</depth></size>"
                  "<object><name>aeroplane</name><difficult>0</difficult>"
                  "<bndbox><xmin>1</xmin><ymin>1</ymin><xmax>20</xmax><ymax>20</ymax>"
                  "</bndbox></object></annotation>")
    xml_path = root / "Annotations" / f"{image_id}.xml"
    xml_path.write_text(annotation)
    dataset = OWDetection(args, root, image_set="smoke_val", dataset="OWDETR",
                          transforms=make_coco_transforms("smoke_val"))
    image, target = dataset[0]
    assert image.shape[0] == 3 and target["labels"].tolist() == [0]
    _, _, ap, *_ = voc_eval(
        [f"{image_id} 0.99 1 1 20 20"], [str(xml_path)], [image_id],
        "aeroplane", known_classes=["aeroplane"])
    assert float(ap) > 0.99
model, criterion, postprocessors, _ = build_model(args, mode="prob")
checkpoint_path = Path(sys.argv[1]) if sys.argv[1] else None
if checkpoint_path is not None:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    assert isinstance(checkpoint, dict) and isinstance(checkpoint.get("epoch"), int)
    state = checkpoint.get("model", checkpoint)
    model_state = model.state_dict()
    compatible_keys = [name for name, value in state.items()
                       if name in model_state and torch.is_tensor(value)
                       and value.shape == model_state[name].shape]
    assert compatible_keys, "T1 checkpoint has no compatible detector parameters"
    incompatible = model.load_state_dict(state, strict=False)
    assert torch.equal(model.state_dict()[compatible_keys[0]].cpu(),
                       state[compatible_keys[0]].cpu())
    print("T1 checkpoint parsed:", checkpoint["epoch"], len(compatible_keys),
          len(incompatible.missing_keys), len(incompatible.unexpected_keys))
model.to("cuda").train()
outputs = model([torch.rand(3, 64, 64, device="cuda")])
required = {"pred_logits", "pred_boxes", "pred_obj", "pred_features", "aux_outputs"}
assert required <= set(outputs)
targets = [{"labels": torch.tensor([0], device="cuda"),
            "boxes": torch.tensor([[0.5, 0.5, 0.25, 0.25]], device="cuda")}]
losses = criterion(outputs, targets)
weighted = sum(losses[name] * criterion.weight_dict[name]
               for name in losses if name in criterion.weight_dict)
assert torch.isfinite(weighted)
weighted.backward()
results = postprocessors["bbox"](outputs, torch.tensor([[64, 64]], device="cuda"))
assert len(results) == 1 and torch.isfinite(results[0]["boxes"]).all()
torch.cuda.synchronize()
chosen = "compiled" if PROB_MSDA_AVAILABLE else "PyTorch fallback"
assert dispatch_counts["compiled" if PROB_MSDA_AVAILABLE else "fallback"] > 0
assert dispatch_counts["fallback" if PROB_MSDA_AVAILABLE else "compiled"] == 0
print("OWOD_MSDA_RESULT=" + json.dumps({
    "available": PROB_MSDA_AVAILABLE,
    "backend": chosen,
    "dispatch_counts": dispatch_counts,
}, sort_keys=True))
print("MSDA backend:", chosen)
print("PROB CUDA model/loss/evaluator smoke: PASS")
"""
checkpoint_for_smoke = Path(DRIVE_ROOT) / CHECKPOINT_RELATIVE
smoke_checkpoint = str(checkpoint_for_smoke) if checkpoint_for_smoke.is_file() else ""
prob_smoke = subprocess.run(
    [sys.executable, "-c", prob_smoke_code, smoke_checkpoint],
    cwd=PROB, capture_output=True, text=True, check=False)
if prob_smoke.returncode != 0:
    print(prob_smoke.stdout)
    print(prob_smoke.stderr)
    print("Environment preflight: FAIL")
    raise RuntimeError("Pinned PROB failed its real CUDA model/evaluator smoke test")
_smoke_rows = [line.removeprefix("OWOD_MSDA_RESULT=")
               for line in prob_smoke.stdout.splitlines()
               if line.startswith("OWOD_MSDA_RESULT=")]
if not _smoke_rows:
    print(prob_smoke.stdout)
    print("Environment preflight: FAIL")
    raise RuntimeError("Real PROB smoke passed without an authoritative MSDA result")
MSDA_SMOKE_RESULT = json.loads(_smoke_rows[-1])
PROB_MSDA_AVAILABLE = bool(MSDA_SMOKE_RESULT["available"])
PROB_MSDA_BACKEND = MSDA_SMOKE_RESULT["backend"]
assert PROB_MSDA_BACKEND == ("compiled" if PROB_MSDA_AVAILABLE else "PyTorch fallback")
if (MSDA_BUILD_ATTEMPTED and MSDA_BUILD_RETURN_CODE == 0
        and not PROB_MSDA_AVAILABLE):
    MSDA_BUILD_MARKER.parent.mkdir(parents=True, exist_ok=True)
    MSDA_BUILD_MARKER.write_text(json.dumps({
        "fingerprint": _msda_fingerprint,
        "backend": PROB_MSDA_BACKEND,
        "reason": "extension built but PROB selected and fully verified fallback",
    }, indent=2), encoding="utf-8")
PROB_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=PROB)
assert PROB_SHA == PROB_COMMIT
ENVIRONMENT_PREFLIGHT_OK = True
runtime = POSTBUILD_PROB_MSDA
print("=" * 60)
print("OWOD ENVIRONMENT PREFLIGHT")
print("=" * 60)
for label, value in (
    ("Runtime Python", runtime.get("python")),
    ("Torch", runtime.get("torch")),
    ("Torchvision", runtime.get("torchvision")),
    ("Torch CUDA", runtime.get("torch_cuda")),
    ("CUDA available", runtime.get("cuda_available")),
    ("GPU", runtime.get("gpu")),
    ("NumPy", runtime.get("numpy")),
    ("SciPy", runtime.get("scipy")),
    ("sklearn", runtime.get("sklearn")),
    ("Pillow", runtime.get("pillow")),
    ("matplotlib", runtime.get("matplotlib")),
    ("pandas", runtime.get("pandas")),
    ("einops", runtime.get("einops")),
    ("pycocotools", runtime.get("pycocotools")),
    ("OWL SHA", OWL_SHA),
    ("PROB SHA", PROB_SHA),
    ("Raw MSDA extension import", "PASS" if RAW_MSDA.get("ok") else "FAIL"),
    ("PROB MSDA_AVAILABLE", PROB_MSDA_AVAILABLE),
    ("MSDA backend", PROB_MSDA_BACKEND),
    ("PROB CUDA model/loss/evaluator smoke", "PASS"),
    ("Environment preflight", "PASS"),
):
    print(f"{label}: {value}")
print("=" * 60)


In [ ]:
# 4 — Prepare the canonical OWOD data root and deterministic shared test split
from concurrent.futures import ThreadPoolExecutor

DATA = Path("/content/data/OWOD")
WORK = DRIVE / "work"
CHECKPOINT = DRIVE / CHECKPOINT_RELATIVE
DATA.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
(DATA / "ImageSets" / "OWDETR").mkdir(parents=True, exist_ok=True)

POOL_ARCHIVE = ROOT / "data" / "staging" / "owdetr_pool_annotations.tar.gz"
TEST_ARCHIVE = ROOT / "data" / "staging" / "owdetr_test_annotations.tar.gz"
REPLAY_ARCHIVE = ROOT / "data" / "staging" / "owdetr_replay_annotations.tar.gz"
for _archive in (POOL_ARCHIVE, TEST_ARCHIVE, REPLAY_ARCHIVE):
    assert _archive.is_file(), f"Missing committed archive: {_archive}"


def extract_committed_archive(source, target):
    target = Path(target).resolve()
    with tarfile.open(source) as handle:
        for member in handle.getmembers():
            destination = (target / member.name).resolve()
            assert destination == target or target in destination.parents, member.name
        try:
            handle.extractall(target, filter="data")
        except TypeError:  # older Python, after the path traversal check above
            handle.extractall(target)


for _archive in (POOL_ARCHIVE, TEST_ARCHIVE, REPLAY_ARCHIVE):
    extract_committed_archive(_archive, DATA)

chain = protocol.build_chain(N_TASKS)
declared = [task.new_class for task in chain[1:]]
subset = evaluation_subset.from_archive(
    TEST_ARCHIVE, declared, seed=SEED,
    remainder_multiplier=EVAL_REMAINDER_RATIO, max_per_class=EVAL_MAX_PER_CLASS,
)
TEST_SET = evaluation_subset.SHARED_TEST_SET
evaluation_subset.write_image_set(
    DATA / "ImageSets" / "OWDETR" / f"{TEST_SET}.txt", subset)

candidate_index = json.loads(
    (ROOT / "data" / "reference" / "per_image_class_counts.json").read_text(encoding="utf-8"))
replay_index = json.loads(
    (ROOT / "data" / "reference" / "t1_replay_class_counts.json").read_text(encoding="utf-8"))
assert candidate_index and replay_index and (DATA / "Annotations").is_dir()

JPEG = DATA / "JPEGImages"
JPEG.mkdir(parents=True, exist_ok=True)


def fetch_images(image_ids, workers=32):
    image_ids = [str(value) for value in image_ids]
    missing = [i for i in image_ids if not (JPEG / f"{i}.jpg").is_file()]

    def fetch(image_id):
        target = JPEG / f"{image_id}.jpg"
        for split in ("train2017", "val2017"):
            subprocess.run(
                ["curl", "-sfL", "--retry", "3", "--retry-delay", "1", "-o", str(target),
                 f"https://images.cocodataset.org/{split}/{image_id}.jpg"],
                check=False,
            )
            if target.is_file() and target.stat().st_size > 0:
                return
        target.unlink(missing_ok=True)

    if missing:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            list(pool.map(fetch, missing))
    return [i for i in image_ids if (JPEG / f"{i}.jpg").is_file()]


print("Data ready:", len(candidate_index), "candidate images,", len(replay_index),
      "replay images,", len(subset.image_ids), "shared-test images")


In [ ]:
# 5 — Build the exact fingerprint and run a concise fail-closed preflight
import csv
import re
from dataclasses import replace

import torch

base_config = runner.CycleConfig(
    n_tasks=N_TASKS, budget_per_task=BUDGET_PER_TASK,
    rounds_per_task=ROUNDS_PER_TASK,
    candidate_images_per_task=CANDIDATE_IMAGES,
    proposals_per_image=PROPOSALS_PER_IMAGE,
    arm=SELECTION_ARM, labelling_policy=LABELLING_POLICY,
    replay_arm="uniform", replay_reallocate=REPLAY_REALLOCATE,
    replay_protocol_version=3, epochs=EPOCHS,
    learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE,
    n_clusters=N_CLUSTERS, seed=SEED, measure_grouped_recall=True,
)


def expected_fingerprint(replay_arm):
    return replace(base_config, replay_arm=replay_arm).fingerprint()


def fingerprint_value_differences(stored, expected):
    return {
        name: (stored.get(name, "(absent)"), value)
        for name, value in expected.items()
        if name not in stored or stored[name] != value
    }


def fingerprint_differences(path, expected):
    stamp = Path(path) / "config.json"
    if not stamp.exists():
        return {}
    stored = json.loads(stamp.read_text(encoding="utf-8"))
    return fingerprint_value_differences(stored, expected)


def legacy_no_replay_baseline_decision(
    *, workspace_name, stored, expected, completed, integrity_ok,
    no_replay_artifacts,
):
    """Accept one absent field only where replay semantics were inactive."""

    differences = fingerprint_value_differences(stored, expected)
    if not differences:
        return differences, None
    allowed = (
        workspace_name == "random__none"
        and stored.get("replay_arm") == "none"
        and expected.get("replay_arm") == "none"
        and "replay_protocol_version" not in stored
        and set(differences) == {"replay_protocol_version"}
        and completed
        and integrity_ok
        and no_replay_artifacts
    )
    if not allowed:
        return differences, None
    note = {
        "workspace": workspace_name,
        "stored": "absent",
        "normalized_for_compatibility": "no-replay-only",
        "reason": "replay_arm=none; replay protocol inactive",
        "historical_config_modified": False,
    }
    return {}, note


def workspace_problem(path):
    path = Path(path)
    if not path.exists():
        return ""
    completed = []
    for task in [f"t{i}" for i in range(2, N_TASKS + 1)]:
        task_dir = path / f"{task}_{SELECTION_ARM}"
        state, scored = task_dir / "state.json", task_dir / "metrics.json"
        if state.exists() != scored.exists():
            return f"{task_dir} has only one of state.json and metrics.json"
        if state.exists():
            completed.append(task)
    expected_prefix = [f"t{i}" for i in range(2, 2 + len(completed))]
    if completed != expected_prefix:
        return f"non-prefix completed tasks: {completed}"
    results = path / f"results_{SELECTION_ARM}.csv"
    if results.exists():
        with results.open(newline="", encoding="utf-8") as handle:
            recorded = [row["task"] for row in csv.DictReader(handle)]
        if recorded != completed:
            return f"results tasks {recorded} disagree with states {completed}"
    elif completed:
        return f"{completed} have state/metrics but results CSV is missing"
    return ""


def completed_baseline_evidence(path, expected_tasks):
    """Validate completion, evaluator artefacts, and that replay was inactive."""

    integrity_reasons = []
    replay_reasons = []
    problem = workspace_problem(path)
    if problem:
        integrity_reasons.append(problem)
    run = comparison.load_run(path)
    if run is None:
        integrity_reasons.append("completed run cannot be loaded")
        return False, False, integrity_reasons, replay_reasons, run
    if run.tasks != list(expected_tasks):
        integrity_reasons.append(f"tasks {run.tasks} != {list(expected_tasks)}")
    if set(run.per_task_ap) != set(expected_tasks):
        integrity_reasons.append("per-task AP is incomplete")
    elif any(len(values) != 81 for values in run.per_task_ap.values()):
        integrity_reasons.append("per-task AP does not contain 81 classes")
    if set(run.per_task_recall) != set(expected_tasks):
        integrity_reasons.append("raw-detection recall cross-check is incomplete")
    for task in expected_tasks:
        report = run.per_class_checks.get(task, {})
        if not report.get("usable") or not report.get("checks"):
            integrity_reasons.append(f"{task} evaluator vector failed integrity validation")

        state_path = Path(path) / f"{task}_{SELECTION_ARM}" / "state.json"
        try:
            state = json.loads(state_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            replay_reasons.append(f"{task} state is missing or unreadable")
            continue
        diagnostics = state.get("replay_row")
        if not isinstance(diagnostics, dict):
            replay_reasons.append(f"{task} has no replay diagnostics")
            continue
        zero_fields = (
            "requested_objects", "allocated_objects", "delivered_objects",
            "images", "unique_source_images", "from_previous_memory",
            "from_new_task", "evicted", "added",
        )
        nonzero = {
            name: diagnostics.get(name, "(absent)")
            for name in zero_fields
            if diagnostics.get(name, "(absent)") != 0
        }
        if nonzero:
            replay_reasons.append(f"{task} replay diagnostics are not zero: {nonzero}")
        if state.get("exemplars"):
            replay_reasons.append(f"{task} stores exemplar identities")

    return (
        not integrity_reasons,
        not replay_reasons,
        integrity_reasons,
        replay_reasons,
        run,
    )


BASELINE = WORK / BASELINE_NAME
TARGETS = {arm: WORK / f"random__{arm}" for arm in REPLAY_ARMS}
BASELINE_EXPECTED_TASKS = [task.name for task in chain[1:]]
BASELINE_CONFIG_PATH = BASELINE / "config.json"
BASELINE_CONFIG_BYTES = (
    BASELINE_CONFIG_PATH.read_bytes() if BASELINE_CONFIG_PATH.is_file() else None
)
BASELINE_STORED_CONFIG = (
    json.loads(BASELINE_CONFIG_BYTES.decode("utf-8"))
    if BASELINE_CONFIG_BYTES is not None else {}
)
(
    BASELINE_INTEGRITY_OK,
    BASELINE_NO_REPLAY_ARTIFACTS,
    BASELINE_INTEGRITY_REASONS,
    BASELINE_REPLAY_REASONS,
    _baseline_evidence_run,
) = completed_baseline_evidence(BASELINE, BASELINE_EXPECTED_TASKS)
BASELINE_COMPLETED = (
    _baseline_evidence_run is not None
    and _baseline_evidence_run.tasks == BASELINE_EXPECTED_TASKS
)
baseline_diff, LEGACY_BASELINE_COMPATIBILITY = legacy_no_replay_baseline_decision(
    workspace_name=BASELINE.name,
    stored=BASELINE_STORED_CONFIG,
    expected=expected_fingerprint("none"),
    completed=BASELINE_COMPLETED,
    integrity_ok=BASELINE_INTEGRITY_OK,
    no_replay_artifacts=BASELINE_NO_REPLAY_ARTIFACTS,
)
if LEGACY_BASELINE_COMPATIBILITY:
    print("Legacy no-replay baseline:")
    print("replay_protocol_version absent — accepted as semantically irrelevant "
          "because replay_arm=none.")


def compatible_run(run):
    """Project only the accepted no-replay legacy field in memory."""

    if (
        run is not None
        and LEGACY_BASELINE_COMPATIBILITY
        and run.name == BASELINE_NAME
        and "replay_protocol_version" not in run.config
    ):
        projected = dict(run.config)
        projected["replay_protocol_version"] = base_config.replay_protocol_version
        return replace(run, config=projected)
    return run


def load_compatible_run(path):
    return compatible_run(comparison.load_run(path))


def load_compatible_runs(path):
    return {
        name: compatible_run(run)
        for name, run in comparison.load_runs(path).items()
    }


COMPARISON_WORKSPACE = Path("/content/owod_no_replay_compatibility_view")


def assert_historical_baseline_unchanged():
    if BASELINE_CONFIG_BYTES is not None:
        assert BASELINE_CONFIG_PATH.read_bytes() == BASELINE_CONFIG_BYTES, (
            "Historical random__none config.json was modified")


def build_comparison_workspace():
    """Create a local read-only-style projection; never edit Drive provenance."""

    shutil.rmtree(COMPARISON_WORKSPACE, ignore_errors=True)
    COMPARISON_WORKSPACE.mkdir(parents=True)
    for name in comparison.EXPECTED:
        source = WORK / name
        if not source.is_dir():
            continue
        target = COMPARISON_WORKSPACE / name
        target.mkdir()
        for item in source.iterdir():
            destination = target / item.name
            if (
                name == BASELINE_NAME
                and item.name == "config.json"
                and LEGACY_BASELINE_COMPATIBILITY
            ):
                projected = dict(BASELINE_STORED_CONFIG)
                projected["replay_protocol_version"] = (
                    base_config.replay_protocol_version)
                destination.write_text(json.dumps(projected, indent=2), encoding="utf-8")
            else:
                destination.symlink_to(
                    item.resolve(), target_is_directory=item.is_dir())
    assert_historical_baseline_unchanged()
    return COMPARISON_WORKSPACE


def annotate_legacy_comparison(summary_directory):
    summary_path = Path(summary_directory) / "summary.json"
    if LEGACY_BASELINE_COMPATIBILITY and summary_path.is_file():
        payload = json.loads(summary_path.read_text(encoding="utf-8"))
        payload["legacy_baseline_replay_protocol"] = LEGACY_BASELINE_COMPATIBILITY
        summary_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
gpu_memory_match = re.search(r"(\d+)\s*MiB", gpu_probe.stdout)
GPU_MEMORY_MIB = int(gpu_memory_match.group(1)) if gpu_memory_match else 0
DRIVE_FREE_GB = shutil.disk_usage(DRIVE).free / 2**30
LOCAL_FREE_GB = shutil.disk_usage(DATA).free / 2**30
package_probe = pip_check()
prob_bridge = bridge.Bridge(
    prob_root=PROB, data_root=DATA, log_dir=DRIVE / "logs", num_workers=2, seed=SEED)

target_diffs = {arm: fingerprint_differences(path, expected_fingerprint(arm))
                for arm, path in TARGETS.items()}
target_problems = {arm: workspace_problem(path) for arm, path in TARGETS.items()}
checks = {
    "CUDA model smoke": ENVIRONMENT_PREFLIGHT_OK,
    "GPU visible": gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip()),
    "GPU memory >= 14 GiB": GPU_MEMORY_MIB >= 14_000,
    "torch CUDA": bool(torch.cuda.is_available()),
    "package consistency": package_probe.returncode == 0,
    "OWL exact SHA": OWL_SHA == OWL_COMMIT,
    "PROB exact SHA": PROB_SHA == PROB_COMMIT,
    "Drive writable": DRIVE.is_dir() and os.access(DRIVE, os.W_OK),
    "Drive free >= 8 GiB": DRIVE_FREE_GB >= 8.0,
    "local free >= 12 GiB": LOCAL_FREE_GB >= 12.0,
    "checkpoint": CHECKPOINT.is_file(),
    "canonical data root": all((DATA / name).is_dir()
                               for name in ("Annotations", "JPEGImages", "ImageSets")),
    "completed baseline exists": BASELINE_COMPLETED,
    "baseline fingerprint": not baseline_diff,
    "target fingerprints": not any(target_diffs.values()),
    "target artefact integrity": not any(target_problems.values()),
    "Replay Protocol V3": base_config.replay_protocol_version == 3
                          and all(expected_fingerprint(arm)["replay_protocol_version"] == 3
                                  for arm in REPLAY_ARMS),
    "workspace isolation": len({BASELINE.resolve(), *(p.resolve() for p in TARGETS.values())}) == 3
                           and BASELINE.name not in PLANNED_RUNS,
}
try:
    BRIDGE_CHECK = prob_bridge.check()
    checks["PROB bridge flags"] = True
except Exception as error:  # noqa: BLE001
    BRIDGE_CHECK = {"error": str(error)}
    checks["PROB bridge flags"] = False

print(f"{'preflight':30s} status")
print("-" * 40)
for name, passed in checks.items():
    print(f"{name:30s} {'PASS' if passed else 'FAIL'}")
if baseline_diff:
    print("baseline differences:", baseline_diff)
if BASELINE_INTEGRITY_REASONS:
    print("baseline integrity problems:", BASELINE_INTEGRITY_REASONS)
if BASELINE_REPLAY_REASONS:
    print("baseline replay evidence problems:", BASELINE_REPLAY_REASONS)
if any(target_diffs.values()):
    print("target differences:", target_diffs)
if any(target_problems.values()):
    print("target artefact problems:", target_problems)
if package_probe.returncode:
    print("package conflicts:", package_probe.stdout + package_probe.stderr)
failed = [name for name, passed in checks.items() if not passed]
assert not failed, f"PREFLIGHT FAILED: {failed}. No GPU evaluation or training was started."
assert_historical_baseline_unchanged()
PREFLIGHT_OK = True
GPU_NAME = torch.cuda.get_device_name(0)
print("PREFLIGHT PASS —", GPU_NAME, "| torch", torch.__version__, "| CUDA", torch.version.cuda)


In [ ]:
# 6 — Verify the historical baseline, validate/create its anchor, compare before training
EXPECTED_TASKS = BASELINE_EXPECTED_TASKS


def validate_detection_run(run, replay_arm, require_anchor=True, allow_partial=False):
    assert run is not None, f"Missing run random__{replay_arm}"
    wanted_tasks = EXPECTED_TASKS[:len(run.tasks)] if allow_partial else EXPECTED_TASKS
    assert run.tasks and run.tasks == wanted_tasks, (run.name, run.tasks, wanted_tasks)
    assert run.config == expected_fingerprint(replay_arm), f"Fingerprint mismatch: {run.name}"
    assert set(run.per_task_ap) == set(wanted_tasks), run.per_task_ap.keys()
    assert all(len(values) == 81 for values in run.per_task_ap.values())
    assert set(run.per_task_recall) == set(wanted_tasks), run.per_task_recall.keys()
    for task in wanted_tasks:
        report = run.per_class_checks.get(task, {})
        assert report.get("usable") and report.get("checks"), (run.name, task, report)
    if require_anchor:
        assert len(run.anchor_ap) == 81, f"{run.name}: missing/invalid 81-class anchor"
        anchor_report = run.per_class_checks.get("anchor", {})
        assert anchor_report.get("usable") and anchor_report.get("checks"), anchor_report
        per_class = comparison.table_per_class({run.name: run})
        assert len(per_class) == len(protocol.TASK1)
        assert all(row.get(f"{run.name}:forgetting") is not None for row in per_class)
    return run


baseline = load_compatible_run(BASELINE)
validate_detection_run(baseline, "none", require_anchor=False)
ready = fetch_images(subset.image_ids)
assert ready == list(subset.image_ids), "Not every shared-test image downloaded"

anchor_command = [
    sys.executable, str(ROOT / "tools" / "evaluate_anchor.py"),
    "--workspace", str(BASELINE), "--checkpoint", str(CHECKPOINT),
    "--prob-root", str(PROB), "--data-root", str(DATA),
    "--archive", str(TEST_ARCHIVE),
    "--eval-max-per-class", str(EVAL_MAX_PER_CLASS),
    "--eval-remainder-ratio", str(EVAL_REMAINDER_RATIO),
]
_checked([*anchor_command, "--dry-run"])
if not (BASELINE / "anchor_metrics.json").is_file():
    _checked(anchor_command)
    RUN_STATUS[BASELINE_NAME] = "anchor generated; baseline reused"
else:
    _checked(anchor_command)  # re-verifies every input, then preserves the existing anchor
    RUN_STATUS[BASELINE_NAME] = "validated and reused"

baseline = validate_detection_run(load_compatible_run(BASELINE), "none", require_anchor=True)
for _arm, _path in TARGETS.items():
    _existing = comparison.load_run(_path)
    if _existing is not None:
        validate_detection_run(_existing, _arm, require_anchor=True, allow_partial=True)
pre_runs = load_compatible_runs(WORK)
pre_clashes = comparison.compatibility(pre_runs, reference=BASELINE_NAME)
assert not pre_clashes, f"Incompatible existing runs before training: {pre_clashes}"
PRECOMPARE = Path("/content/owod_preflight_comparison")
shutil.rmtree(PRECOMPARE, ignore_errors=True)
_checked([sys.executable, str(ROOT / "tools" / "compare_replay.py"), str(build_comparison_workspace()),
          "--out", str(PRECOMPARE), "--include", ",".join(comparison.EXPECTED),
          "--no-plots"])
assert (PRECOMPARE / "summary.json").is_file()
annotate_legacy_comparison(PRECOMPARE)
assert_historical_baseline_unchanged()
VALIDATION_OUTPUTS[BASELINE_NAME] = str(BASELINE / "anchor_metrics.json")
print("Baseline PASS:", EXPECTED_TASKS, "| anchor/per-class/recall/forgetting validated")


In [ ]:
# 7 — Run or resume random__uniform (420-minute per-run cap)
def completed_depth(path):
    found = comparison.load_run(path)
    return len(found) if found is not None else 0


def run_replay_arm(replay_arm):
    target = TARGETS[replay_arm]
    before = completed_depth(target)
    gpu_minutes_before = float(prob_bridge.cost_report()["total"])
    assert gpu_minutes_before < SESSION_CEILING_MINUTES, (
        f"Session ceiling reached before random__{replay_arm}. Reconnect and Run all; the run resumes safely.")
    per_run_budget = min(TIME_BUDGET_MINUTES, SESSION_CEILING_MINUTES - gpu_minutes_before)
    assert 0 < per_run_budget <= TIME_BUDGET_MINUTES
    rows = runner.run_chain(
        prob_bridge, replace(base_config, replay_arm=replay_arm),
        workspace=target, candidate_index=candidate_index, replay_index=replay_index,
        replay_root=DATA, start_checkpoint=CHECKPOINT, test_set=TEST_SET, chain=chain,
        time_budget_minutes=per_run_budget, prepare_images=fetch_images,
    )
    after = len(rows)
    RUN_STATUS[f"random__{replay_arm}"] = (
        "validated and skipped" if before == len(EXPECTED_TASKS)
        else "resumed and completed" if before > 0 and after == len(EXPECTED_TASKS)
        else "completed" if after == len(EXPECTED_TASKS)
        else f"partial ({after}/{len(EXPECTED_TASKS)}); Run all again to resume"
    )
    return rows


by_arm = {}
by_arm["random__uniform"] = run_replay_arm("uniform")
assert len(by_arm["random__uniform"]) == len(EXPECTED_TASKS), RUN_STATUS["random__uniform"]


In [ ]:
# 8 — Strict Replay Protocol V3 audit for random__uniform
def validate_replay_workspace(replay_arm):
    name = f"random__{replay_arm}"
    path = TARGETS[replay_arm]
    run = validate_detection_run(comparison.load_run(path), replay_arm, require_anchor=True)
    budget = int(replay.ARMS[replay_arm]["total"])
    assert budget == 400
    prior_exemplars = set()
    prior_task_images = set()
    task_audits = []
    for index, task in enumerate(EXPECTED_TASKS):
        state_path = path / f"{task}_{SELECTION_ARM}" / "state.json"
        assert state_path.is_file(), state_path
        state = json.loads(state_path.read_text(encoding="utf-8"))
        diagnostics = state["replay_row"]
        assert (diagnostics["requested_objects"] == diagnostics["allocated_objects"]
                == diagnostics["delivered_objects"] == budget), (task, diagnostics)
        current = {tuple(row) for row in state["exemplars"]}
        assert len(current) == budget
        per_class = comparison.parse_per_class_quota(diagnostics["per_class"])
        assert sum(per_class.values()) == budget
        rebuilt = {}
        for _, class_name, _ in current:
            rebuilt[class_name] = rebuilt.get(class_name, 0) + 1
        assert rebuilt == per_class
        sources = {row[0] for row in current}
        assert sources.isdisjoint(state["previous_task_images"]), f"{task}: replayed current-task source"
        assert diagnostics["images"] == diagnostics["unique_source_images"] == len(sources)
        aliases = {exemplars.alias_id(source) for source in sources}
        assert len(aliases) == len(sources)
        assert all(exemplars.source_id(alias) in sources for alias in aliases)
        if index == 0:
            assert sources <= set(replay_index), f"{task}: source outside canonical old-data pool"
        else:
            assert all(item in prior_exemplars or item[0] in prior_task_images for item in current), (
                f"{task}: exemplar resurrected outside E_(k-1) union L_(k-1)")
        digest = hashlib.sha256(json.dumps(sorted(current), separators=(",", ":")).encode()).hexdigest()
        task_audits.append({
            "task": task, "requested": budget, "allocated": budget, "delivered": budget,
            "objects": len(current), "source_images": len(sources),
            "per_class_total": sum(per_class.values()), "identity_sha256": digest,
            "from_previous_memory": diagnostics["from_previous_memory"],
            "added": diagnostics["added"], "evicted": diagnostics["evicted"],
        })
        prior_exemplars = current
        prior_task_images = set(state["previous_task_images"])
    audit = {
        "schema": "owl_replay_v3_audit_v1", "run": name,
        "config": run.config, "tasks": task_audits,
        "per_class_validated": run.per_class_ap_is_validated,
        "recall_crosschecks": sorted(run.per_task_recall),
        "resume_identity": "state-restored object identities are SHA-256 recorded per task",
    }
    output = path / "replay_v3_audit.json"
    pending = output.with_suffix(".json.pending")
    pending.write_text(json.dumps(audit, indent=2), encoding="utf-8")
    pending.replace(output)
    VALIDATION_OUTPUTS[name] = str(output)
    print(name, "PASS — requested = allocated = delivered = 400 at t2–t6")
    return run


uniform_run = validate_replay_workspace("uniform")


In [ ]:
# 9 — Run or resume random__tail_favouring (only after uniform passes)
by_arm["random__tail_favouring"] = run_replay_arm("tail_favouring")
assert len(by_arm["random__tail_favouring"]) == len(EXPECTED_TASKS), RUN_STATUS["random__tail_favouring"]


In [ ]:
# 10 — Strict Replay Protocol V3 audit for random__tail_favouring
tail_run = validate_replay_workspace("tail_favouring")
all_runs = load_compatible_runs(WORK)
assert list(all_runs) == list(comparison.EXPECTED), list(all_runs)
assert not comparison.compatibility(all_runs, reference=BASELINE_NAME)
assert all(run.tasks == EXPECTED_TASKS for run in all_runs.values())
print("All three runs are complete, protocol-identical, and comparison-ready.")


In [ ]:
# 11 — Generate the full comparison locally, validate it, then persist it to Drive
LOCAL_COMPARISON = Path("/content/owod_comparison_replay_v3_fast_seed0")
PERSISTENT_COMPARISON = DRIVE / "comparisons" / COMPARISON_NAME
shutil.rmtree(LOCAL_COMPARISON, ignore_errors=True)
_checked([sys.executable, str(ROOT / "tools" / "compare_replay.py"), str(build_comparison_workspace()),
          "--out", str(LOCAL_COMPARISON), "--include", ",".join(comparison.EXPECTED)])

table_stems = (
    "depth", "table1_task_comparison", "table2_delta_vs_baseline",
    "table3_tail_vs_uniform", "table4_per_class", "table5_replay_composition",
    "table6_cost",
)
required_outputs = {"summary.json"}
for stem in table_stems:
    required_outputs.update({f"{stem}.csv", f"{stem}.md", f"{stem}.tex"})
required_outputs.update({
    f"figure_{letter}_{suffix}.{extension}"
    for letter, suffix in (
        ("a", "group_ap"), ("b", "forgetting"), ("c", "new_class_ap"),
        ("d", "forgetting_vs_frequency"), ("e", "replay_allocation"),
        ("f", "forgetting_vs_anchor"),
    ) for extension in ("png", "pdf")
})
missing = sorted(name for name in required_outputs if not (LOCAL_COMPARISON / name).is_file())
assert not missing, f"Comparison output incomplete: {missing}"
annotate_legacy_comparison(LOCAL_COMPARISON)
assert_historical_baseline_unchanged()
summary = json.loads((LOCAL_COMPARISON / "summary.json").read_text(encoding="utf-8"))
assert set(summary["runs"]) == set(comparison.EXPECTED)
assert not summary["missing"] and not summary["compatibility_clashes"]
if LEGACY_BASELINE_COMPATIBILITY:
    assert summary["legacy_baseline_replay_protocol"] == LEGACY_BASELINE_COMPATIBILITY

PERSISTENT_COMPARISON.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_COMPARISON, PERSISTENT_COMPARISON, dirs_exist_ok=True)
missing_persistent = sorted(
    name for name in required_outputs if not (PERSISTENT_COMPARISON / name).is_file())
assert not missing_persistent, missing_persistent
VALIDATION_OUTPUTS["comparison"] = str(PERSISTENT_COMPARISON)
print("Comparison PASS:", len(required_outputs), "validated files copied to", PERSISTENT_COMPARISON)


In [ ]:
# 12 — Final audit summary; success marker is printed only after every assertion passed
EXPERIMENT_COMPLETE = True
print("=" * 78)
print("FINAL OWOD REPLAY V3 SUMMARY")
print("=" * 78)
print("OWL SHA: ", OWL_SHA)
print("PROB SHA:", PROB_SHA)
print("GPU:     ", GPU_NAME)
print("torch:   ", torch.__version__, "| CUDA:", torch.version.cuda,
      "| MSDA:", PROB_MSDA_BACKEND)
if LEGACY_BASELINE_COMPATIBILITY:
    print("legacy baseline compatibility:", LEGACY_BASELINE_COMPATIBILITY)
print("runs:")
for name in comparison.EXPECTED:
    print(f"  {name:25s} {RUN_STATUS[name]}")
print("validations / outputs:")
for name, output in VALIDATION_OUTPUTS.items():
    print(f"  {name:25s} {output}")
print("session elapsed minutes:", round((time.monotonic() - SESSION_STARTED) / 60.0, 1),
      "/", SESSION_CEILING_MINUTES)
print("metered GPU minutes:", round(float(prob_bridge.cost_report()["total"]), 1),
      "/", SESSION_CEILING_MINUTES)
print("EXPERIMENT COMPLETE")
